In [ ]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd

In [ ]:
import requests
import pandas as pd
from datetime import datetime

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

all_records = []

start_year = datetime.now().year - 5
end_year = datetime.now().year

for year in range(start_year, end_year + 1):
    for month in range(1, 13):

        start_date = f"{year}-{month:02d}-01"

        if month == 12:
            end_date = f"{year+1}-01-01"
        else:
            end_date = f"{year}-{month+1:02d}-01"

        params = {
            "format": "geojson",
            "starttime": start_date,
            "endtime": end_date,
            "minmagnitude": 3
        }

        response = requests.get(url, params=params)

        if response.status_code != 200:
            print(f"Failed for {start_date}")
            continue

        data = response.json()

        for f in data["features"]:
            p = f["properties"]
            g = f["geometry"]["coordinates"]

            all_records.append({
                "id": f.get("id"),
                "time": pd.to_datetime(p.get("time"), unit="ms"),
                "updated": pd.to_datetime(p.get("updated"), unit="ms"),

                "latitude": g[1] if len(g) > 1 else None,
                "longitude": g[0] if len(g) > 0 else None,
                "depth_km": g[2] if len(g) > 2 else None,

                "mag": p.get("mag"),
                "magType": p.get("magType"),
                "place": p.get("place"),
                "status": p.get("status"),
                "tsunami": p.get("tsunami"),
                "sig": p.get("sig"),
                "mmi": p.get("mmi"),
                "alert": p.get("alert"),
                "felt": p.get("felt"),
                "cdi": p.get("cdi"),

                "net": p.get("net"),
                "code": p.get("code"),
                "ids": p.get("ids"),
                "sources": p.get("sources"),
                "types": p.get("types"),

                "nst": p.get("nst"),
                "dmin": p.get("dmin"),
                "rms": p.get("rms"),
                "gap": p.get("gap"),
                "type": p.get("type")
            })

# Create DataFrame
df = pd.DataFrame(all_records)

# Display summary
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Show first 5 rows
print(df.head())

# Show column names
print("\nColumns:")
print(df.columns.tolist())


Rows: 113443
Columns: 26
           id                    time                 updated  latitude  \
0  us6000ddi8 2021-01-31 23:20:49.923 2021-04-16 19:02:44.040  -31.7493   
1  us6000dev6 2021-01-31 23:08:17.161 2021-04-16 19:03:47.040  -15.4902   
2  us6000dev5 2021-01-31 22:54:19.760 2021-04-16 19:03:47.040   19.7529   
3  us6000ddhs 2021-01-31 22:06:00.832 2021-04-16 19:02:43.040   28.1524   
4  us6000dev4 2021-01-31 21:51:14.016 2021-04-16 19:03:46.040   71.3212   

   longitude  depth_km  mag magType  \
0   -68.9337     17.27  4.7     mwr   
1  -177.2052    426.71  4.1      mb   
2   121.3159     46.73  4.7      mb   
3    57.2570     10.00  4.9      mb   
4    -3.7578     10.00  4.0      mb   

                                               place    status  ...  net  \
0        29 km SW of Villa Basilio Nievas, Argentina  reviewed  ...   us   
1                                        Fiji region  reviewed  ...   us   
2                    103 km SW of Basco, Philippines  reviewe

In [ ]:
df.columns

Index(['id', 'time', 'updated', 'latitude', 'longitude', 'depth_km', 'mag',
       'magType', 'place', 'status', 'tsunami', 'sig', 'mmi', 'alert', 'felt',
       'cdi', 'net', 'code', 'ids', 'sources', 'types', 'nst', 'dmin', 'rms',
       'gap', 'type'],
      dtype='object')

In [ ]:
df['place']

0               29 km SW of Villa Basilio Nievas, Argentina
1                                               Fiji region
2                           103 km SW of Basco, Philippines
3                                   114 km N of M?n?b, Iran
4         184 km ENE of Olonkinbyen, Svalbard and Jan Mayen
                                ...                        
113438                           southeast of Easter Island
113439                            Izu Islands, Japan region
113440                            Izu Islands, Japan region
113441     63 km N of Charlotte Amalie, U.S. Virgin Islands
113442                              Macquarie Island region
Name: place, Length: 113443, dtype: object

In [ ]:
import re

In [ ]:
df['country'] = df['place'].apply(lambda x: re.search(r', ([^,]+)$', str(x)).group(1) if re.search(r', ([^,]+)$', str(x)) else str(x).strip())

In [ ]:
df[['place','country']]

,place,country
0,"29 km SW of Villa Basilio Nievas, Argentina",Argentina
1,Fiji region,Fiji region
2,"103 km SW of Basco, Philippines",Philippines
3,"114 km N of M?n?b, Iran",Iran
4,"184 km ENE of Olonkinbyen, Svalbard and Jan Mayen",Svalbard and Jan Mayen
...,...,...
113438,southeast of Easter Island,southeast of Easter Island
113439,"Izu Islands, Japan region",Japan region
113440,"Izu Islands, Japan region",Japan region
113441,"63 km N of Charlotte Amalie, U.S. Virgin Islands",U.S. Virgin Islands


In [ ]:
df.describe(include='all')

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,code,ids,sources,types,nst,dmin,rms,gap,type,country
count,113443,113443,113443,113443.000000,113443.000000,113443.000000,113443.000000,113443,113443,113443,...,113443,113443,113443,113443,85681.000000,109468.000000,113424.000000,110421.000000,113443,113443
unique,113443,NaN,NaN,NaN,NaN,NaN,NaN,17,62590,2,...,113443,113443,369,579,NaN,NaN,NaN,NaN,11,531
top,us7000spra,NaN,NaN,NaN,NaN,NaN,NaN,mb,South Sandwich Islands region,reviewed,...,7000spra,",us7000spra,",",us,",",origin,phase-data,",NaN,NaN,NaN,NaN,earthquake,Alaska
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,80391,3118,113268,...,1,1,82717,86274,NaN,NaN,NaN,NaN,112582,13849
mean,NaN,2023-09-20 01:33:29.194007808,2024-01-07 00:21:07.464819968,12.108838,7.304676,71.932315,4.256422,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,45.423956,3.185841,0.657044,122.477148,NaN,NaN
min,NaN,2021-01-01 00:14:07.580000,2021-01-01 10:15:32.601000,-84.493200,-179.999700,-3.740000,3.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,8.000000,NaN,NaN
25%,NaN,2022-04-19 00:19:39.119000064,2022-08-13 21:51:27.040000,-14.705700,-113.923450,10.000000,4.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,20.000000,0.828000,0.490000,75.000000,NaN,NaN
50%,NaN,2023-09-25 18:36:06.796000,2024-01-18 20:55:53.203000064,14.997000,23.860400,21.600000,4.300000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,32.000000,1.808000,0.650000,113.000000,NaN,NaN
75%,NaN,2025-03-01 01:18:35.972000,2025-06-14 18:26:46.040000,39.069000,130.520050,71.146500,4.600000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,56.000000,3.726000,0.820000,156.000000,NaN,NaN
max,NaN,2026-06-15 12:36:10.120000,2026-06-15 14:32:18.040000,87.375200,179.999400,683.578000,8.800000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,619.000000,62.558000,3.400000,359.000000,NaN,NaN


In [ ]:
df.select_dtypes(include='object').values.T

array([['us6000ddi8', 'us6000dev6', 'us6000dev5', ..., 'us7000sqy4',
        'pr71518453', 'us7000spra'],
       ['mwr', 'mb', 'mb', ..., 'mb', 'md', 'mb'],
       ['29 km SW of Villa Basilio Nievas, Argentina', 'Fiji region',
        '103 km SW of Basco, Philippines', ...,
        'Izu Islands, Japan region',
        '63 km N of Charlotte Amalie, U.S. Virgin Islands',
        'Macquarie Island region'],
       ...,
       [',dyfi,moment-tensor,origin,phase-data,', ',origin,phase-data,',
        ',origin,phase-data,', ..., ',origin,phase-data,',
        ',origin,phase-data,', ',origin,phase-data,'],
       ['earthquake', 'earthquake', 'earthquake', ..., 'earthquake',
        'earthquake', 'earthquake'],
       ['Argentina', 'Fiji region', 'Philippines', ..., 'Japan region',
        'U.S. Virgin Islands', 'Macquarie Island region']],
      shape=(12, 113443), dtype=object)

In [ ]:
print(df.head())

           id                    time                 updated  latitude  \
0  us6000ddi8 2021-01-31 23:20:49.923 2021-04-16 19:02:44.040  -31.7493   
1  us6000dev6 2021-01-31 23:08:17.161 2021-04-16 19:03:47.040  -15.4902   
2  us6000dev5 2021-01-31 22:54:19.760 2021-04-16 19:03:47.040   19.7529   
3  us6000ddhs 2021-01-31 22:06:00.832 2021-04-16 19:02:43.040   28.1524   
4  us6000dev4 2021-01-31 21:51:14.016 2021-04-16 19:03:46.040   71.3212   

   longitude  depth_km  mag magType  \
0   -68.9337     17.27  4.7     mwr   
1  -177.2052    426.71  4.1      mb   
2   121.3159     46.73  4.7      mb   
3    57.2570     10.00  4.9      mb   
4    -3.7578     10.00  4.0      mb   

                                               place    status  ...      code  \
0        29 km SW of Villa Basilio Nievas, Argentina  reviewed  ...  6000ddi8   
1                                        Fiji region  reviewed  ...  6000dev6   
2                    103 km SW of Basco, Philippines  reviewed  ...  60

In [ ]:
if 'alert' in df.columns:
    df['alert'] = df['alert'].astype(str).str.lower().replace('nan', pd.NA)
    string_cols = ['magType', 'status', 'type', 'net', 'sources', 'types']
for col in string_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).fillna('').str.strip().str.lower()

In [ ]:
df[string_cols].dtypes

magType    object
status     object
type       object
net        object
sources    object
types      object
dtype: object

In [ ]:
df['alert'].isna().sum()

np.int64(0)

In [ ]:
df.isna().sum()

id                0
time              0
updated           0
latitude          0
longitude         0
depth_km          0
mag               0
magType           0
place             0
status            0
tsunami           0
sig               0
mmi          103675
alert             0
felt          96042
cdi           96042
net               0
code              0
ids               0
sources           0
types             0
nst           27762
dmin           3975
rms              19
gap            3022
type              0
country           0
dtype: int64

In [ ]:
df[['nst','dmin', 'rms', 'gap', 'mmi', 'felt', 'cdi']].isna().sum()

nst      27762
dmin      3975
rms         19
gap       3022
mmi     103675
felt     96042
cdi      96042
dtype: int64

In [ ]:
df['nst'] = pd.to_numeric(df['nst'], errors='coerce')
df['nst'] = df['nst'].fillna(df['nst'].median())
df['mmi'] = pd.to_numeric(df['mmi'], errors='coerce')
df['mmi'] = df['mmi'].fillna(df['mmi'].median())
df['felt'] = pd.to_numeric(df['felt'], errors='coerce')
df['felt'] = df['felt'].fillna(df['felt'].median())
df['cdi'] = pd.to_numeric(df['cdi'], errors='coerce')
df['cdi'] = df['cdi'].fillna(df['cdi'].median())
df['dmin'] = pd.to_numeric(df['dmin'], errors='coerce')
df['dmin'] = df['dmin'].fillna(df['dmin'].median())
df['rms'] = pd.to_numeric(df['rms'], errors='coerce')
df['rms'] = df['rms'].fillna(df['rms'].median())
df['gap'] = pd.to_numeric(df['gap'], errors='coerce')
df['gap'] = df['gap'].fillna(df['gap'].median())

In [ ]:
df[['nst','dmin', 'rms', 'gap', 'mmi', 'felt', 'cdi']].isna().sum()

nst     0
dmin    0
rms     0
gap     0
mmi     0
felt    0
cdi     0
dtype: int64

In [ ]:
df.isna().sum()

id           0
time         0
updated      0
latitude     0
longitude    0
depth_km     0
mag          0
magType      0
place        0
status       0
tsunami      0
sig          0
mmi          0
alert        0
felt         0
cdi          0
net          0
code         0
ids          0
sources      0
types        0
nst          0
dmin         0
rms          0
gap          0
type         0
country      0
dtype: int64

In [ ]:
df['time']

0        2021-01-31 23:20:49.923
1        2021-01-31 23:08:17.161
2        2021-01-31 22:54:19.760
3        2021-01-31 22:06:00.832
4        2021-01-31 21:51:14.016
                   ...          
113438   2026-06-01 03:40:11.053
113439   2026-06-01 03:34:30.871
113440   2026-06-01 02:51:50.510
113441   2026-06-01 00:33:28.750
113442   2026-06-01 00:08:41.694
Name: time, Length: 113443, dtype: datetime64[ns]

In [ ]:
df['time'].dt.year

0         2021
1         2021
2         2021
3         2021
4         2021
          ... 
113438    2026
113439    2026
113440    2026
113441    2026
113442    2026
Name: time, Length: 113443, dtype: int32

In [ ]:
df['year'] = df['time'].dt.year

In [ ]:
df['time'].dt.month

0         1
1         1
2         1
3         1
4         1
         ..
113438    6
113439    6
113440    6
113441    6
113442    6
Name: time, Length: 113443, dtype: int32

In [ ]:
df['month'] = df['time'].dt.month

In [ ]:
df['time'].dt.day

0         31
1         31
2         31
3         31
4         31
          ..
113438     1
113439     1
113440     1
113441     1
113442     1
Name: time, Length: 113443, dtype: int32

In [ ]:
df['day'] = df['time'].dt.day

In [ ]:
df['time'].dt.day_name()

0         Sunday
1         Sunday
2         Sunday
3         Sunday
4         Sunday
           ...  
113438    Monday
113439    Monday
113440    Monday
113441    Monday
113442    Monday
Name: time, Length: 113443, dtype: object

In [ ]:
df['day_of_week'] = df['time'].dt.day_name()

In [ ]:
df.shape

(113443, 31)

In [ ]:
df.isna().sum()

id             0
time           0
updated        0
latitude       0
longitude      0
depth_km       0
mag            0
magType        0
place          0
status         0
tsunami        0
sig            0
mmi            0
alert          0
felt           0
cdi            0
net            0
code           0
ids            0
sources        0
types          0
nst            0
dmin           0
rms            0
gap            0
type           0
country        0
year           0
month          0
day            0
day_of_week    0
dtype: int64

In [ ]:
df['depth_km']

0          17.27
1         426.71
2          46.73
3          10.00
4          10.00
           ...  
113438     10.00
113439     11.00
113440     10.00
113441     34.17
113442     10.00
Name: depth_km, Length: 113443, dtype: float64

In [ ]:
import numpy as np
np.where(df['depth_km'] < 70, 'shallow', 'deep')

array(['shallow', 'deep', 'shallow', ..., 'shallow', 'shallow', 'shallow'],
      shape=(113443,), dtype='<U7')

In [ ]:
df['depth_category'] = np.where(df['depth_km'] < 70, 'shallow', 'deep')

In [ ]:
df['depth_category']

0         shallow
1            deep
2         shallow
3         shallow
4         shallow
           ...   
113438    shallow
113439    shallow
113440    shallow
113441    shallow
113442    shallow
Name: depth_category, Length: 113443, dtype: object

In [ ]:
df.shape

(113443, 32)

In [ ]:
df['mag']

0         4.70
1         4.10
2         4.70
3         4.90
4         4.00
          ... 
113438    5.30
113439    5.70
113440    4.30
113441    3.39
113442    4.90
Name: mag, Length: 113443, dtype: float64

In [ ]:
conditions = [(df['mag'] >= 6.0) & (df['mag'] < 7.0),(df['mag'] >= 7.0)]
choices = ['Strong', 'Destructive']
df['strong_destructive_flag'] = np.select(conditions, choices, default='Not Strong/Destructive')


In [ ]:
df['strong_destructive_flag']

0         Not Strong/Destructive
1         Not Strong/Destructive
2         Not Strong/Destructive
3         Not Strong/Destructive
4         Not Strong/Destructive
                   ...          
113438    Not Strong/Destructive
113439    Not Strong/Destructive
113440    Not Strong/Destructive
113441    Not Strong/Destructive
113442    Not Strong/Destructive
Name: strong_destructive_flag, Length: 113443, dtype: object

In [ ]:
df.shape

NameError: name 'df' is not defined

In [ ]:
df.to_csv("earthquakes_data.csv", index=False)

NameError: name 'df' is not defined

In [ ]:
df=pd.read_csv('earthquakes_data.csv')
df.head(10)

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,rms,gap,type,country,year,month,day,day_of_week,depth_category,strong_destructive_flag
0,us6000ddi8,2021-01-31 23:20:49.923,2021-04-16 19:02:44.040,-31.7493,-68.9337,17.27,4.70,mwr,"29 km SW of Villa Basilio Nievas, Argentina",reviewed,...,0.82,42.0,earthquake,Argentina,2021,1,31,Sunday,shallow,Not Strong/Destructive
1,us6000dev6,2021-01-31 23:08:17.161,2021-04-16 19:03:47.040,-15.4902,-177.2052,426.71,4.10,mb,Fiji region,reviewed,...,0.29,64.0,earthquake,Fiji region,2021,1,31,Sunday,deep,Not Strong/Destructive
2,us6000dev5,2021-01-31 22:54:19.760,2021-04-16 19:03:47.040,19.7529,121.3159,46.73,4.70,mb,"103 km SW of Basco, Philippines",reviewed,...,0.69,106.0,earthquake,Philippines,2021,1,31,Sunday,shallow,Not Strong/Destructive
3,us6000ddhs,2021-01-31 22:06:00.832,2021-04-16 19:02:43.040,28.1524,57.2570,10.00,4.90,mb,"114 km N of M?n?b, Iran",reviewed,...,0.61,71.0,earthquake,Iran,2021,1,31,Sunday,shallow,Not Strong/Destructive
4,us6000dev4,2021-01-31 21:51:14.016,2021-04-16 19:03:46.040,71.3212,-3.7578,10.00,4.00,mb,"184 km ENE of Olonkinbyen, Svalbard and Jan Mayen",reviewed,...,0.50,65.0,earthquake,Svalbard and Jan Mayen,2021,1,31,Sunday,shallow,Not Strong/Destructive
5,pr2021031019,2021-01-31 21:34:54.690,2021-04-16 19:02:43.040,18.9996,-65.4121,46.00,3.55,md,"76 km NNE of Luquillo, Puerto Rico",reviewed,...,0.31,305.0,earthquake,Puerto Rico,2021,1,31,Sunday,shallow,Not Strong/Destructive
6,pr2021031018,2021-01-31 21:26:30.180,2021-01-31 21:52:15.880,19.0250,-65.4221,30.00,3.27,md,"78 km NNE of Luquillo, Puerto Rico",reviewed,...,0.37,305.0,earthquake,Puerto Rico,2021,1,31,Sunday,shallow,Not Strong/Destructive
7,pr2021031020,2021-01-31 21:14:31.020,2021-04-16 19:02:43.040,19.0436,-65.3590,28.00,3.48,md,"82 km N of Culebra, Puerto Rico",reviewed,...,0.37,309.0,earthquake,Puerto Rico,2021,1,31,Sunday,shallow,Not Strong/Destructive
8,us6000dev3,2021-01-31 20:57:55.804,2021-04-16 19:03:46.040,5.6376,126.7150,18.68,4.30,mb,"99 km SE of Pondaguitan, Philippines",reviewed,...,0.49,139.0,earthquake,Philippines,2021,1,31,Sunday,shallow,Not Strong/Destructive
9,us6000dev2,2021-01-31 20:54:42.248,2021-04-16 19:03:46.040,5.8436,126.5404,75.53,4.20,mb,"69 km SE of Pondaguitan, Philippines",reviewed,...,0.58,196.0,earthquake,Philippines,2021,1,31,Sunday,deep,Not Strong/Destructive


In [ ]:
pip install sqlalchemy psycopg2-binary

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.1 MB 1.2 MB/s eta 0:00:02
   -------------- ------------------------- 0.8/2.1 MB 1.5 MB/s eta 0:00:01
   ------------------------ --------------- 1.3/2.1 MB 1.6 MB/s eta 0:00:01
   ---------------------------------- ----- 1.8/2.1 MB 1.7 MB/s eta 0:00:01
   ---------------------------------------  2.1/2.1 MB 1.8 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 1.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.8 MB 1.2 MB/s eta 0:00:02
   --------------- ------------------------ 1.0/2.8 MB 1.5 MB/s eta 0:00:02
   --------------- ------------------------ 1.0/2.8 MB 1.5 MB/s eta 0:00:02
   -------------------------- ----------


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from sqlalchemy import create_engine
username='postgres'
password='pearlraj10'
port=5432
database_name='earthquake_db'
host_name='localhost'
engine=create_engine(f'postgresql+psycopg2://{username}:{password}@{host_name}:{port}/{database_name}')

In [ ]:
df.to_sql('earthquakes', con=engine, if_exists='replace', index=False,chunksize=10000,method='multi')

113443

# **1. Top 10 Strongest Earthquakes**

In [ ]:
query = """
SELECT
    id,
    time,
    place,
    country,
    mag,
    depth_km,
    strong_destructive_flag
FROM earthquakes
ORDER BY mag DESC
LIMIT 10;
"""

top10 = pd.read_sql(query, engine)


In [ ]:
top10

,id,time,place,country,mag,depth_km,strong_destructive_flag
0,us6000qw60,2025-07-29 23:24:52.483,"2025 Kamchatka Peninsula, Russia Earthquake",Russia Earthquake,8.8,35.000,Destructive
1,ak0219neiszm,2021-07-29 06:15:49.188,"2021 Chignik, Alaska Earthquake",Alaska Earthquake,8.2,35.000,Destructive
2,us7000dflf,2021-03-04 19:28:33.178,"2021 Kermadec Islands, New Zealand Earthquake",New Zealand Earthquake,8.1,28.930,Destructive
3,us6000f53e,2021-08-12 18:35:17.231,2021 South Sandwich Islands Earthquake,2021 South Sandwich Islands Earthquake,8.1,22.790,Destructive
4,us6000jllz,2023-02-06 01:17:34.342,"Pazarcik earthquake, Kahramanmaras earthquake ...",Kahramanmaras earthquake sequence,7.8,10.000,Destructive
5,us7000srb1,2026-06-07 23:37:41.803,"26 km SW of Kablalan, Philippines",Philippines,7.8,55.193,Destructive
6,us7000qx2g,2025-09-18 18:58:14.939,"140 km E of Petropavlovsk-Kamchatsky, Russia",Russia,7.8,27.000,Destructive
7,us7000pn9s,2025-03-28 06:20:52.715,"2025 Mandalay, Burma (Myanmar) Earthquake",Burma (Myanmar) Earthquake,7.7,10.000,Destructive
8,us6000dg77,2021-02-10 13:19:55.530,southeast of the Loyalty Islands,southeast of the Loyalty Islands,7.7,10.000,Destructive
9,us6000kd0n,2023-05-19 02:57:03.172,southeast of the Loyalty Islands,southeast of the Loyalty Islands,7.7,18.053,Destructive


# **2. Top 10 Deepest Earthquakes**

In [ ]:
query = """
SELECT
    id,
    time,
    place,
    country,
    depth_km,
    mag
FROM earthquakes
ORDER BY depth_km DESC
LIMIT 10;
"""

top10_deepest = pd.read_sql(query, engine)

In [ ]:
top10_deepest

,id,time,place,country,depth_km,mag
0,us6000sp3p,2026-04-02 16:59:17.870,"206 km ENE of Sola, Vanuatu",Vanuatu,683.578,4.0
1,us6000k2db,2023-04-01 18:09:17.114,"208 km ENE of Sola, Vanuatu",Vanuatu,681.238,4.0
2,us7000kxdn,2023-09-18 15:35:27.270,Vanuatu region,Vanuatu region,675.265,4.2
3,us6000mivr,2024-03-08 22:42:23.713,Fiji region,Fiji region,671.043,4.2
4,us6000rk66,2025-10-29 17:07:34.157,"205 km ESE of Levuka, Fiji",Fiji,669.556,4.8
5,us6000f2w3,2021-08-01 22:12:46.340,"283 km SE of Levuka, Fiji",Fiji,669.460,4.0
6,us7000q1jk,2025-05-10 02:03:16.756,"299 km E of Levuka, Fiji",Fiji,667.237,4.2
7,us7000s0st,2026-02-14 08:27:48.418,"205 km ESE of Levuka, Fiji",Fiji,667.197,4.3
8,us6000dhfx,2021-02-13 16:26:42.487,south of the Fiji Islands,south of the Fiji Islands,664.740,4.5
9,us7000he64,2022-06-01 18:41:01.389,"279 km ESE of Labasa, Fiji",Fiji,664.700,4.3


# **3. Year with Most Earthquakes**

In [ ]:
query = """
SELECT
    id,
    time,
    place,
    country,
    depth_km,
    mag
FROM earthquakes
WHERE depth_km < 50
  AND mag > 7.5
ORDER BY mag DESC;
"""

shallow_strong_eq = pd.read_sql(query, engine)


In [ ]:
shallow_strong_eq

,id,time,place,country,depth_km,mag
0,us6000qw60,2025-07-29 23:24:52.483,"2025 Kamchatka Peninsula, Russia Earthquake",Russia Earthquake,35.000,8.8
1,ak0219neiszm,2021-07-29 06:15:49.188,"2021 Chignik, Alaska Earthquake",Alaska Earthquake,35.000,8.2
2,us6000f53e,2021-08-12 18:35:17.231,2021 South Sandwich Islands Earthquake,2021 South Sandwich Islands Earthquake,22.790,8.1
3,us7000dflf,2021-03-04 19:28:33.178,"2021 Kermadec Islands, New Zealand Earthquake",New Zealand Earthquake,28.930,8.1
4,us7000qx2g,2025-09-18 18:58:14.939,"140 km E of Petropavlovsk-Kamchatsky, Russia",Russia,27.000,7.8
5,us6000jllz,2023-02-06 01:17:34.342,"Pazarcik earthquake, Kahramanmaras earthquake ...",Kahramanmaras earthquake sequence,10.000,7.8
6,us6000dg77,2021-02-10 13:19:55.530,southeast of the Loyalty Islands,southeast of the Loyalty Islands,10.000,7.7
7,us7000pn9s,2025-03-28 06:20:52.715,"2025 Mandalay, Burma (Myanmar) Earthquake",Burma (Myanmar) Earthquake,10.000,7.7
8,us6000kd0n,2023-05-19 02:57:03.172,southeast of the Loyalty Islands,southeast of the Loyalty Islands,18.053,7.7
9,us6000rtdt,2025-12-08 14:15:09.896,"2025 Aomori Prefecture, Japan Earthquake",Japan Earthquake,40.720,7.6


In [ ]:
df.columns

Index(['id', 'time', 'updated', 'latitude', 'longitude', 'depth_km', 'mag',
       'magType', 'place', 'status', 'tsunami', 'sig', 'mmi', 'alert', 'felt',
       'cdi', 'net', 'code', 'ids', 'sources', 'types', 'nst', 'dmin', 'rms',
       'gap', 'type', 'country', 'year', 'month', 'day', 'day_of_week',
       'depth_category', 'strong_destructive_flag'],
      dtype='object')

# **4. Most Active Reporting Network**
**Continent column not available in earthquake dataset**

In [ ]:
query = """
SELECT
    country,
    AVG(depth_km) AS avg_depth_km
FROM earthquakes
GROUP BY country
ORDER BY avg_depth_km DESC;
"""

avg_depth_country = pd.read_sql(query, engine)


In [ ]:
avg_depth_country

,country,avg_depth_km
0,Peru-Brazil border region,605.390000
1,North Korea,577.230750
2,Fiji region,539.216355
3,eastern Russia-northeastern China border region,520.242000
4,Fiji,519.388300
...,...,...
526,North Carolina,0.100000
527,Tennessee-Virginia border region,0.000000
528,Iowa,0.000000
529,Indonesia,0.000000


# **5. Average magnitude per magnitude type (magType)**

In [ ]:
query = """
SELECT
    "magType",
    ROUND(AVG(mag)::numeric, 2) AS avg_magnitude,
    COUNT(*) AS earthquake_count
FROM earthquakes
GROUP BY "magType"
ORDER BY avg_magnitude DESC;
"""

avg_magtype = pd.read_sql(query, engine)
avg_magtype


,magType,avg_magnitude,earthquake_count
0,mwc,6.15,2
1,ms_20,5.80,1
2,mwb,5.80,30
3,mww,5.37,6966
4,mwp,5.25,1
5,ms_vx,4.60,2
6,mb,4.42,80391
7,mwr,4.33,2724
8,mh,4.10,1
9,mw,3.94,653


# **6. Year with most earthquakes**

In [ ]:
query = """
SELECT
    year,
    COUNT(*) AS total_earthquakes
FROM earthquakes
GROUP BY year
ORDER BY total_earthquakes DESC
LIMIT 1;
"""

year_most_eq = pd.read_sql(query, engine)
year_most_eq

,year,total_earthquakes
0,2025,22818


# **7. Month with highest number of earthquakes**

In [ ]:
query = """
SELECT
    month,
    COUNT(*) AS total_earthquakes
FROM earthquakes
GROUP BY month
ORDER BY total_earthquakes DESC
LIMIT 1;
"""

month_most_eq = pd.read_sql(query, engine)
month_most_eq

,month,total_earthquakes
0,3,10906


# **8. Day of week with most earthquakes**

In [ ]:
query = """
SELECT
    day_of_week,
    COUNT(*) AS total_earthquakes
FROM earthquakes
GROUP BY day_of_week
ORDER BY total_earthquakes DESC
LIMIT 1;
"""

day_most_eq = pd.read_sql(query, engine)
day_most_eq

,day_of_week,total_earthquakes
0,Friday,16384


# **9. Count of earthquakes per hour of day**

In [ ]:
query = """
SELECT
    DATE_PART('hour', CAST(time AS TIMESTAMP)) AS hour_of_day,
    COUNT(*) AS total_earthquakes
FROM earthquakes
GROUP BY hour_of_day
ORDER BY hour_of_day;
"""

hourly_eq = pd.read_sql(query, engine)
hourly_eq

,hour_of_day,total_earthquakes
0,0.0,4705
1,1.0,4977
2,2.0,4822
3,3.0,5039
4,4.0,4996
5,5.0,4655
6,6.0,4578
7,7.0,4721
8,8.0,4712
9,9.0,4678


# **10.   Most active reporting network (net)**

In [ ]:
query = """
SELECT
    net,
    COUNT(*) AS total_earthquakes
FROM earthquakes
GROUP BY net
ORDER BY total_earthquakes DESC
LIMIT 1;
"""

most_active_net = pd.read_sql(query, engine)
most_active_net

,net,total_earthquakes
0,us,99189


# **11.  Top 5 places with highest casualties**

In [ ]:
query = """
SELECT
    place,
    COUNT(*) AS earthquake_count
FROM earthquakes
GROUP BY place
ORDER BY earthquake_count DESC
LIMIT 5;
"""

top_places = pd.read_sql(query, engine)
top_places

,place,earthquake_count
0,South Sandwich Islands region,3118
1,south of the Fiji Islands,1990
2,Kermadec Islands region,1910
3,Fiji region,1294
4,southeast of the Loyalty Islands,1069


# **12.  Total estimated economic loss per continent**
**Continent column not available in earthquake dataset**

# **13.  Average economic loss by alert level**

In [ ]:
query = """
SELECT
    alert,
    COUNT(*) AS earthquake_count,
    ROUND(AVG(mag)::numeric, 2) AS avg_magnitude
FROM earthquakes
GROUP BY alert
ORDER BY avg_magnitude DESC;
"""

alert_analysis = pd.read_sql(query, engine)
alert_analysis

,alert,earthquake_count,avg_magnitude
0,red,24,6.69
1,orange,24,6.32
2,yellow,129,6.00
3,green,4470,5.38
4,none,108796,4.21


# **14.  Count of reviewed vs automatic earthquakes (status)**

In [ ]:
query = """
SELECT
    status,
    COUNT(*) AS earthquake_count
FROM earthquakes
GROUP BY status
ORDER BY earthquake_count DESC;
"""

status_count = pd.read_sql(query, engine)
status_count

,status,earthquake_count
0,reviewed,113268
1,automatic,175


# **15.  Count by earthquake type (type)**

In [ ]:
query = """
SELECT
    type,
    COUNT(*) AS earthquake_count
FROM earthquakes
GROUP BY type
ORDER BY earthquake_count DESC;
"""

type_count = pd.read_sql(query, engine)
type_count

,type,earthquake_count
0,earthquake,112582
1,mining explosion,818
2,ice quake,13
3,volcanic eruption,12
4,other event,5
5,landslide,4
6,experimental explosion,3
7,mine collapse,3
8,quarry blast,2
9,explosion,1


# **16.  Number of earthquakes by data type (types)**

In [ ]:
query = """
SELECT
    types,
    COUNT(*) AS earthquake_count
FROM earthquakes
GROUP BY types
ORDER BY earthquake_count DESC;
"""

types_count = pd.read_sql(query, engine)
types_count

,types,earthquake_count
0,",origin,phase-data,",86274
1,",dyfi,origin,phase-data,",8680
2,",origin,phase-data,shakemap,",2183
3,",earthquake-name,origin,phase-data,",2172
4,",moment-tensor,origin,phase-data,",1361
...,...,...
574,",dyfi,focal-mechanism,origin,phase-data,shakemap,",1
575,",dyfi,focal-mechanism,impact-link,moment-tenso...",1
576,",dyfi,focal-mechanism,moment-tensor,nearby-cit...",1
577,",dyfi,general-text,losspager,moment-tensor,ori...",1


# **17.  Average RMS and gap per continent**
**Continent column not available in earthquake dataset**

# **18.  Events with high station coverage (nst > threshold)**

In [ ]:
query = """
SELECT
    id,
    time,
    place,
    mag,
    nst
FROM earthquakes
WHERE nst > 100
ORDER BY nst DESC;
"""

high_station_events = pd.read_sql(query, engine)
high_station_events

,id,time,place,mag,nst
0,us6000m12f,2024-01-02 01:17:31.568,"11 km W of Anamizu, Japan",5.40,619.0
1,us6000qzfl,2025-08-09 08:01:41.017,"120 km ENE of Ozernovskiy, Russia",5.00,566.0
2,us7000rluk,2026-01-01 06:46:54.742,"111 km N of Yakutat, Alaska",5.70,516.0
3,us7000pvtr,2025-04-29 14:53:37.897,Macquarie Island region,6.80,475.0
4,usd001097k,2023-12-11 18:35:59.877,"49 km WNW of San Antonio de los Cobres, Argentina",5.50,466.0
...,...,...,...,...,...
7431,us6000r2xq,2025-08-22 02:16:18.246,2025 Southern Drake Passage Earthquake,7.50,101.0
7432,ci40865184,2024-08-07 04:09:56.760,"24 km SW of Lamont, CA",5.22,101.0
7433,us7000q9pb,2025-06-30 06:36:20.640,"293 km SSE of Port Blair, India",4.90,101.0
7434,nc73872965,2023-04-13 07:42:10.810,"1km NNW of The Geysers, CA",3.88,101.0


# **19.  Number of tsunamis triggered per year**

In [ ]:
query = """
SELECT
    year,
    COUNT(*) AS tsunami_count
FROM earthquakes
WHERE tsunami = 1
GROUP BY year
ORDER BY year;
"""

tsunami_per_year = pd.read_sql(query, engine)
tsunami_per_year

,year,tsunami_count
0,2021,114
1,2022,136
2,2023,119
3,2024,114
4,2025,142
5,2026,34


# **20.  Count earthquakes by alert levels (red, orange, etc.)**

In [ ]:
query = """
SELECT
    alert,
    COUNT(*) AS earthquake_count
FROM earthquakes
GROUP BY alert
ORDER BY earthquake_count DESC;
"""

alert_count = pd.read_sql(query, engine)
alert_count

,alert,earthquake_count
0,none,108796
1,green,4470
2,yellow,129
3,red,24
4,orange,24


# **21.Find the top 5 countries with the highest average magnitude of earthquakes in the past 5 years**

In [ ]:
query = """
SELECT
    country,
    ROUND(AVG(mag)::numeric, 2) AS avg_magnitude,
    COUNT(*) AS earthquake_count
FROM earthquakes
GROUP BY country
HAVING COUNT(*) > 0
ORDER BY avg_magnitude DESC
LIMIT 5;
"""

top5_countries = pd.read_sql(query, engine)
top5_countries

,country,avg_magnitude,earthquake_count
0,Russia Earthquake,8.10,2
1,2021 South Sandwich Islands Earthquake,8.10,1
2,New Zealand Earthquake,8.10,1
3,Burma (Myanmar) Earthquake,7.70,1
4,Kahramanmaras earthquake sequence,7.65,2


# **22.Find countries that have experienced both shallow and deep earthquakes within the same month**

In [ ]:
query = """
SELECT
    country,
    year,
    month
FROM earthquakes
GROUP BY country, year, month
HAVING
    SUM(CASE WHEN depth_km < 70 THEN 1 ELSE 0 END) > 0
    AND
    SUM(CASE WHEN depth_km > 300 THEN 1 ELSE 0 END) > 0
ORDER BY country, year, month;
"""

result = pd.read_sql(query, engine)
result

,country,year,month
0,Afghanistan,2022,10
1,Argentina,2021,2
2,Argentina,2021,5
3,Argentina,2021,6
4,Argentina,2021,10
...,...,...,...
852,Wallis and Futuna,2025,6
853,Wallis and Futuna,2025,7
854,Wallis and Futuna,2025,8
855,Wallis and Futuna,2025,12


# **23.Compute the year-over-year growth rate in the total number of earthquakes globally**

In [ ]:
query = """
WITH yearly_counts AS (
    SELECT
        year,
        COUNT(*) AS total_earthquakes
    FROM earthquakes
    GROUP BY year
)
SELECT
    year,
    total_earthquakes,
    LAG(total_earthquakes) OVER (ORDER BY year) AS previous_year_count,
    ROUND(
        (
            (total_earthquakes - LAG(total_earthquakes) OVER (ORDER BY year))
            * 100.0
            / LAG(total_earthquakes) OVER (ORDER BY year)
        )::numeric,
        2
    ) AS yoy_growth_rate
FROM yearly_counts
ORDER BY year;
"""

yoy_growth = pd.read_sql(query, engine)
yoy_growth

,year,total_earthquakes,previous_year_count,yoy_growth_rate
0,2021,21926,NaN,NaN
1,2022,20208,21926.0,-7.84
2,2023,20830,20208.0,3.08
3,2024,18659,20830.0,-10.42
4,2025,22818,18659.0,22.29
5,2026,9002,22818.0,-60.55


# **24. List the 3 most seismically active regions by combining both frequency and average magnitude**

In [ ]:
query = """
SELECT
    country,
    COUNT(*) AS earthquake_count,
    ROUND(AVG(mag)::numeric, 2) AS avg_magnitude,
    ROUND((COUNT(*) * AVG(mag))::numeric, 2) AS activity_score
FROM earthquakes
WHERE country IS NOT NULL
GROUP BY country
ORDER BY activity_score DESC
LIMIT 3;
"""

top_regions = pd.read_sql(query, engine)
top_regions

,country,earthquake_count,avg_magnitude,activity_score
0,Alaska,13849,3.47,48087.71
1,Indonesia,8726,4.50,39233.10
2,Russia,6193,4.48,27769.10


# **25. For each country, calculate the average depth of earthquakes within ±5° latitude range of the equator**

In [ ]:
query = """
SELECT
    country,
    COUNT(*) AS earthquake_count,
    ROUND(AVG(depth_km)::numeric, 2) AS avg_depth_km
FROM earthquakes
WHERE latitude BETWEEN -5 AND 5
  AND country IS NOT NULL
GROUP BY country
ORDER BY avg_depth_km DESC;
"""

equator_depth = pd.read_sql(query, engine)
equator_depth

,country,earthquake_count,avg_depth_km
0,Celebes Sea,11,452.79
1,Banda Sea,11,339.38
2,Philippines,518,119.56
3,Bismarck Sea,6,106.02
4,Papua New Guinea,1167,71.42
5,Peru,94,64.20
6,Ecuador,307,63.60
7,Indonesia,5730,62.45
8,Peru-Ecuador border region,5,59.68
9,northern Peru,2,55.07


# **26. Identify countries having the highest ratio of shallow to deep earthquakes**

In [ ]:
query = """
SELECT
    country,
    SUM(CASE WHEN depth_km < 70 THEN 1 ELSE 0 END) AS shallow_count,
    SUM(CASE WHEN depth_km > 300 THEN 1 ELSE 0 END) AS deep_count,
    ROUND(
        (
            SUM(CASE WHEN depth_km < 70 THEN 1 ELSE 0 END)::numeric
            / NULLIF(SUM(CASE WHEN depth_km > 300 THEN 1 ELSE 0 END), 0)
        ),
        2
    ) AS shallow_deep_ratio
FROM earthquakes
WHERE country IS NOT NULL
GROUP BY country
HAVING SUM(CASE WHEN depth_km > 300 THEN 1 ELSE 0 END) > 0
ORDER BY shallow_deep_ratio DESC
LIMIT 10;
"""

ratio_df = pd.read_sql(query, engine)
ratio_df

,country,shallow_count,deep_count,shallow_deep_ratio
0,China,1358,1,1358.00
1,Peru,709,1,709.00
2,Guam,468,1,468.00
3,Afghanistan,257,1,257.00
4,New Zealand,1404,11,127.64
5,Solomon Islands,796,8,99.50
6,Russia,5285,99,53.38
7,Vanuatu,1137,28,40.61
8,south of the Kermadec Islands,655,18,36.39
9,Japan,4132,127,32.54


# **27. Find the average magnitude difference between earthquakes with tsunami alerts and those without**

In [ ]:
query = """
SELECT
    tsunami,
    ROUND(AVG(mag)::numeric, 2) AS avg_magnitude,
    COUNT(*) AS earthquake_count
FROM earthquakes
GROUP BY tsunami
ORDER BY tsunami DESC;
"""

tsunami_mag = pd.read_sql(query, engine)
tsunami_mag

,tsunami,avg_magnitude,earthquake_count
0,1,5.42,659
1,0,4.25,112784


# **28. Using the gap and rms columns, identify events with the lowest data reliability (highest average error margins)**

In [ ]:
query = """
SELECT
    id,
    time,
    place,
    mag,
    gap,
    rms,
    ROUND(((gap + rms) / 2.0)::numeric, 2) AS reliability_error_score
FROM earthquakes
WHERE gap IS NOT NULL
  AND rms IS NOT NULL
ORDER BY reliability_error_score DESC
LIMIT 10;
"""

low_reliability_events = pd.read_sql(query, engine)
low_reliability_events

,id,time,place,mag,gap,rms,reliability_error_score
0,pr71519528,2026-06-11 12:21:59.730,"95 km ENE of Cruz Bay, U.S. Virgin Islands",3.31,359.00,0.1400,179.57
1,nn00897599,2025-05-14 12:02:09.568,"60 km NE of Valmy, Nevada",3.00,358.18,0.1601,179.17
2,pr2024051000,2024-02-20 05:23:09.140,"77 km NW of Sandy Ground Village, Anguilla",3.58,358.00,0.1600,179.08
3,pr71508583,2026-02-23 19:35:44.000,"86 km WNW of Sandy Ground Village, Anguilla",3.17,356.00,0.2200,178.11
4,us7000denb,2021-02-14 19:26:12.202,"Andreanof Islands, Aleutian Islands, Alaska",3.30,355.00,0.3400,177.67
5,aka2026kbzasg,2026-05-22 19:57:17.293,"181 km E of Atka, Alaska",3.50,354.00,0.7000,177.35
6,pr2022227000,2022-08-15 01:27:45.500,"59 km ENE of Cruz Bay, U.S. Virgin Islands",3.63,353.00,0.6100,176.81
7,pr71489653,2025-07-18 23:19:06.230,"53 km SSW of Boca de Yuma, Dominican Republic",3.13,352.00,0.1800,176.09
8,pr2022289000,2022-10-16 20:43:51.670,"78 km E of Cruz Bay, U.S. Virgin Islands",3.48,352.00,0.0700,176.04
9,pr2021003002,2021-01-03 01:55:58.430,"87 km WNW of The Bottom, Bonaire, Saint Eustat...",3.53,351.00,0.4400,175.72


# **29. Find pairs of consecutive earthquakes (by time) that occurred within 50 km of each other and within 1 hour**
**Continent column not available in earthquake dataset**

# **30. Determine the regions with the highest frequency of deep-focus earthquakes (depth > 300 km)**

In [ ]:
query = """
SELECT
    country,
    COUNT(*) AS deep_earthquake_count,
    ROUND(AVG(depth_km)::numeric, 2) AS avg_depth_km
FROM earthquakes
WHERE depth_km > 300
  AND country IS NOT NULL
GROUP BY country
ORDER BY deep_earthquake_count DESC
LIMIT 10;
"""

deep_focus_regions = pd.read_sql(query, engine)
deep_focus_regions

,country,deep_earthquake_count,avg_depth_km
0,south of the Fiji Islands,1670,522.85
1,Fiji region,1248,556.71
2,Fiji,1200,565.61
3,Tonga,606,471.56
4,Indonesia,285,457.87
5,Japan region,249,448.56
6,Timor Leste,212,445.31
7,Wallis and Futuna,164,407.98
8,Kermadec Islands region,162,391.65
9,Northern Mariana Islands,128,434.79


# **streamlit**

In [ ]:
pip install streamlit

  Using cached pillow-12.2.0-cp312-cp312-win_amd64.whl.metadata (9.0 kB)
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import streamlit as st

In [ ]:
st.set_page_config(
    page_title="Earthquake Analytics Dashboard",
    layout="wide"
)

st.title("Earthquake Data Analysis Dashboard")


2026-06-18 03:04:45.672 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-18 03:04:45.674 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-18 03:04:45.676 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-18 03:04:45.679 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [ ]:
total_eq = pd.read_sql(
    "SELECT COUNT(*) total FROM earthquakes",
    engine
).iloc[0,0]

st.metric(
    "Total Earthquakes",
    f"{total_eq:,}"
)

In [ ]:
st.map(df[['latitude','longitude']])

In [ ]:
import plotly.express as px

fig = px.pie(
    df,
    names='alert'
)

st.plotly_chart(fig)

In [ ]:
page = st.sidebar.radio(
    "Navigation",
    [
        "Dashboard",
        "Query Analysis",
        "Map View"
    ]
)